# Inspect the Pile windows dataset

Load the balanced Pile subset built by `data_utils/build_pile_windows.py`
and show a few sample rows.

In [1]:
from datasets import load_from_disk
from transformers import AutoTokenizer
from collections import Counter

DATA_DIR = "/orfeo/scratch/dssc/zenocosini/pile_gemma2b_100M_windows/merged"
ds = load_from_disk(DATA_DIR)
print(ds)
print("columns:", ds.column_names)
print("num rows:", len(ds))
print("total tokens:", len(ds) * len(ds[0]['token_ids']))

/u/dssc/zenocosini/decomposing-activations-local-geometry/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dataset({
    features: ['text', 'subset', 'token_ids', 'window_start', 'window_end', 'doc_len'],
    num_rows: 328964
})
columns: ['text', 'subset', 'token_ids', 'window_start', 'window_end', 'doc_len']
num rows: 328964
total tokens: 84214784


In [9]:
ds[0]["doc_len"]

11813

## Per-subset counts

In [2]:
counts = Counter(ds['subset'])
for s, n in sorted(counts.items()):
    print(f"{s:30s} {n:>7d}  ({n*256:>10,d} tokens)")

pile-arxiv                       22977  ( 5,882,112 tokens)
pile-dm_mathematics              22977  ( 5,882,112 tokens)
pile-enron_emails                13769  ( 3,524,864 tokens)
pile-europarl                    22977  ( 5,882,112 tokens)
pile-freelaw                     22977  ( 5,882,112 tokens)
pile-github                      20840  ( 5,335,040 tokens)
pile-gutenberg_pg-19              4000  ( 1,024,000 tokens)
pile-hackernews                  16846  ( 4,312,576 tokens)
pile-nih_exporter                22977  ( 5,882,112 tokens)
pile-philpapers                  22977  ( 5,882,112 tokens)
pile-pile-cc                     20097  ( 5,144,832 tokens)
pile-pubmed_abstracts            16152  ( 4,134,912 tokens)
pile-pubmed_central              22977  ( 5,882,112 tokens)
pile-stackexchange               22977  ( 5,882,112 tokens)
pile-ubuntu_irc-broken           13168  ( 3,371,008 tokens)
pile-uspto_backgrounds           22977  ( 5,882,112 tokens)
pile-wikipedia_en                17299  

## One sample per subset

Shows the stored `text` (doc capped at window end) and the decoded window tokens.

In [7]:
tok = AutoTokenizer.from_pretrained("google/gemma-2b")

seen = set()
for row in ds:
    s = row['subset']
    if s in seen:
        continue
    seen.add(s)
    window_text = tok.decode(row['token_ids'], skip_special_tokens=False)
    print("=" * 80)
    print(f"subset       : {s}")
    print(f"window       : [{row['window_start']}, {row['window_end']})")
    print(f"doc_len      : {row['doc_len']} tokens")
    print(f"text len     : {len(row['text'])} chars")
    print(f"--- text head ---\n{row['text'][:300]}")
    print(f"--- text tail ---\n{row['text'][-300:]}")
    print(f"--- window (decoded from token_ids) ---\n{window_text[:400]}")
    if len(seen) == len(counts):
        break

subset       : pile-arxiv
window       : [6311, 6567)
doc_len      : 11813 tokens
text len     : 20097 chars
--- text head ---
---
abstract: 'The purpose of this article is to study the problem of finding sharp lower bounds for the norm of the product of polynomials in the ultraproducts of Banach spaces $(X_i)_{\mathfrak U}$. We show that, under certain hypotheses, there is a strong relation between this problem and the sam
--- text tail ---
rod_{j=1}^n \sum_{t=1}^{m_j} (JT\psi_{j,t}(x))^{r_{j,t}} \right| \nonumber \\
&=& \left|\prod_{j=1}^n \sum_{t=1}^{m_j}((JT)^*\hat{x}(\psi_{j,t}))^{r_{j,t}}\right|\nonumber\end{aligned}$$

Since $(JT)^*\hat{x}\in M^*$, $\Vert (JT)^*\hat{x}\Vert =\Vert JT \Vert \Vert x \Vert \leq \Vert J \Vert \Vert T
--- window (decoded from token_ids) ---
 product $\Vert \prod_{j=1}^n \bar{Q}_j \Vert$. Let $x=(x_i)_{\mathfrak U}$ be any point in $B_{(X_i)_{\mathfrak U}}$. Then, we have $$\begin{aligned}
 \left|\prod_{j=1}^n \bar{Q}_j(x)\right| &=& \left|\prod_{j=1}

## Sanity checks

In [10]:
# All windows should be exactly 256 tokens.
lens = [len(r['token_ids']) for r in ds.select(range(1000))]
print("window lengths (first 1000):", set(lens))

# No special IDs should appear in any window.
special_ids = set(tok.all_special_ids or [])
for t in (tok.bos_token_id, tok.eos_token_id, tok.pad_token_id, tok.unk_token_id):
    if t is not None:
        special_ids.add(t)
bad = 0
for r in ds.select(range(2000)):
    if special_ids & set(r['token_ids']):
        bad += 1
print(f"rows with special tokens in window (of 2000): {bad}")

window lengths (first 1000): {256}
rows with special tokens in window (of 2000): 0
